<a href="https://colab.research.google.com/github/phineas-pta/gg_colab_AI_playground/blob/main/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# Text generation with LLM

In [ ]:
!nvidia-smi

In [ ]:
%pip install -qU transformers bitsandbytes

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = "llmfan46/gemma-4-12B-it-qat-q4_0-unquantized-uncensored-heretic"
PROCESSOR = AutoProcessor.from_pretrained(MODEL_ID)
MODEL = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, device_map="auto", quantization_config=BitsAndBytesConfig(load_in_4bit=True))

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant. Provide the answer in plain text without Markdown. Do not output anything else."""

@torch.inference_mode()
def generate(prompt):
	messages = [
		{"role": "system", "content": SYSTEM_PROMPT},
		{"role": "user", "content": prompt},
	]
	inputs = PROCESSOR.apply_chat_template(messages, tokenize=True, return_dict=True, return_tensors="pt", add_generation_prompt=True, enable_thinking=False).to(MODEL.device)
	input_len = inputs.input_ids.shape[-1]
	outputs = MODEL.generate(**inputs, max_new_tokens=2**10)
	response = PROCESSOR.decode(outputs[0][input_len:], skip_special_tokens=False)
	return PROCESSOR.parse_response(response)["content"]

In [ ]:
print(generate("Translate to Vietnamese: The doctor will see you now."))